In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import json
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax
from tqdm import tqdm

# ======================================================
# Load MedQA dataset (JSONL-safe with robust answer parsing)
# ======================================================
def load_jsonl(file_path):
    """Reads JSONL file line by line safely."""
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

medqa_path = "/content/drive/MyDrive/MedQA/test.jsonl"
medqa_data = load_jsonl(medqa_path)

answer_map = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}

questions, labels, choices = [], [], []

for item in medqa_data:
    q = item.get("question", "")
    opts = item.get("options", ["A", "B", "C", "D", "E"])
    ans = item.get("answer", "").strip()

    # Case 1: Answer is a letter label
    if ans in answer_map:
        label = answer_map[ans]
    # Case 2: Answer matches one of the option texts
    elif any(ans.strip().lower() == o.strip().lower() or ans.strip().lower() in o.strip().lower() for o in opts):
        for i, o in enumerate(opts):
            if ans.strip().lower() == o.strip().lower() or ans.strip().lower() in o.strip().lower():
                label = i
                break
        else:
            label = 0
    # Case 3: Missing or malformed answer
    else:
        label = 0

    questions.append(q)
    labels.append(label)
    choices.append(opts)

labels_np = np.array(labels)

# ======================================================
# Utility Functions
# ======================================================
def compute_ece(probs, labels, n_bins=15):
    confidences = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    accuracies = (predictions == labels)
    ece = 0.0
    bins = np.linspace(0, 1, n_bins + 1)
    for i in range(n_bins):
        mask = (confidences > bins[i]) & (confidences <= bins[i + 1])
        if np.any(mask):
            ece += np.abs(np.mean(confidences[mask]) - np.mean(accuracies[mask])) * np.mean(mask)
    return ece

def compute_nll(logits, labels):
    return F.cross_entropy(torch.tensor(logits), torch.tensor(labels)).item()

def compute_cus(uncertainties):
    return float(np.mean(uncertainties))

def compute_zti(cus_scores, eps=1e-6):
    if len(cus_scores) < 2:
        return 1.0
    mean, std = np.mean(cus_scores), np.std(cus_scores)
    max_cus = max(np.max(cus_scores), eps)
    zti = 1 - (np.log1p(abs(mean - std)) / np.log1p(max_cus))
    return float(np.clip(zti, 0, 1))

# ======================================================
# Model Setup
# ======================================================
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ======================================================
# Deep Ensemble Evaluation
# ======================================================
N_ENSEMBLES = 5
print(f"\nRunning Deep Ensemble with {N_ENSEMBLES} ClinicalBERT models...\n")

ensemble_models = [
    AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=5)
    for _ in range(N_ENSEMBLES)
]

ensemble_preds = []

for model_idx, model in enumerate(ensemble_models):
    print(f"Evaluating model {model_idx + 1}/{N_ENSEMBLES}...")
    model.eval()
    probs_list = []
    for q, opts in tqdm(zip(questions, choices), total=len(questions), leave=False):
        text = q + " " + " ".join(opts)
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
        with torch.no_grad():
            logits = model(**inputs).logits
            probs = softmax(logits.squeeze().cpu().numpy(), axis=-1)
        probs_list.append(probs)
    ensemble_preds.append(np.array(probs_list))

ensemble_preds = np.array(ensemble_preds)  # [N_ENSEMBLES, N, num_classes]
mean_probs = np.mean(ensemble_preds, axis=0)
uncertainties = np.std(ensemble_preds, axis=0) / (mean_probs + 1e-6)

ece_de = compute_ece(mean_probs, labels_np)
nll_de = compute_nll(mean_probs, labels_np)
cus_de = compute_cus(uncertainties)
zti_de = compute_zti([cus_de])

# ======================================================
# SWAG Evaluation (Stochastic Weight Averaging Gaussian)
# ======================================================
class SWAGModel(nn.Module):
    """Simplified SWAG implementation for inference-only evaluation."""
    def __init__(self, base_model, swag_scale=0.05, n_samples=10):
        super().__init__()
        self.base_model = base_model
        self.swag_scale = swag_scale
        self.n_samples = n_samples

    def sample_weights(self):
        """Generate stochastic weights for Gaussian posterior approximation."""
        for param in self.base_model.parameters():
            if param.requires_grad:
                noise = torch.randn_like(param) * self.swag_scale
                param.data.add_(noise)

    def predict(self, inputs):
        probs_list = []
        for _ in range(self.n_samples):
            self.sample_weights()
            with torch.no_grad():
                logits = self.base_model(**inputs).logits
                probs = softmax(logits.squeeze().cpu().numpy(), axis=-1)
            probs_list.append(probs)
        return np.mean(probs_list, axis=0), np.std(probs_list, axis=0)

print("\nRunning SWAG-based ClinicalBERT evaluation...\n")
base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=5)
swag_model = SWAGModel(base_model, swag_scale=0.03, n_samples=10)

swag_preds, swag_uncs = [], []

for q, opts in tqdm(zip(questions, choices), total=len(questions), leave=False):
    text = q + " " + " ".join(opts)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    mean_pred, unc = swag_model.predict(inputs)
    swag_preds.append(mean_pred)
    swag_uncs.append(unc)

swag_preds = np.array(swag_preds)
swag_uncs = np.array(swag_uncs)

ece_swag = compute_ece(swag_preds, labels_np)
nll_swag = compute_nll(swag_preds, labels_np)
cus_swag = compute_cus(swag_uncs)
zti_swag = compute_zti([cus_swag])

# ======================================================
# Collect and Save Results
# ======================================================
results = pd.DataFrame([
    {
        "Model Variant": f"ClinicalBERT Deep Ensemble (N={N_ENSEMBLES})",
        "ECE ↓": round(ece_de, 4),
        "NLL ↓": round(nll_de, 4),
        "CUS ↓": round(cus_de, 4),
        "ZTI ↑": round(zti_de, 4)
    },
    {
        "Model Variant": "ClinicalBERT + SWAG (Stochastic Weight Averaging Gaussian)",
        "ECE ↓": round(ece_swag, 4),
        "NLL ↓": round(nll_swag, 4),
        "CUS ↓": round(cus_swag, 4),
        "ZTI ↑": round(zti_swag, 4)
    }
])

print("\n==== Uncertainty Quantification Results (Deep Ensemble + SWAG) ====\n")
print(results.to_string(index=False))
results.to_csv("uq_results.csv", index=False)

print("\nResults saved to uq_results.csv ")



Running Deep Ensemble with 5 ClinicalBERT models...



Some weights of BertForSequenceClassification were not initialized from the model checkpoint at emilyalsentzer/Bio_ClinicalBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at emilyalsentzer/Bio_ClinicalBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at emilyalsentzer/Bio_ClinicalBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the mod

Evaluating model 1/5...


Evaluating model 2/5...


Evaluating model 3/5...


Evaluating model 4/5...


Evaluating model 5/5...



Running SWAG-based ClinicalBERT evaluation...



Some weights of BertForSequenceClassification were not initialized from the model checkpoint at emilyalsentzer/Bio_ClinicalBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
 80%|████████  | 1023/1273 [5:08:36<1:24:21, 20.24s/it]